In [1]:
from lsst.daf.butler import Butler
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import SymLogNorm
from matplotlib.lines import Line2D

import matplotlib as mpl

# --- Paper figure style ---
# All text is 12 pt. It appears at 12 pt in the paper only if each PDF is saved at
# exactly COLUMN_WIDTH and included with \includegraphics[width=\columnwidth]{...}.
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "figure.titlesize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
    "legend.frameon": False,
    "axes.linewidth": 0.6,
    "lines.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.minor.size": 2.0,
    "ytick.minor.size": 2.0,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "standard",  # keep the saved PDF at exactly figsize (no tight cropping)
})

# LaTeX \columnwidth in inches: put \showthe\columnwidth in the paper, read the value
# in pt from the log, and divide by 72.27.
COLUMN_WIDTH = 3.5
FIG_SIZE = (COLUMN_WIDTH, 0.75 * COLUMN_WIDTH)  # default single-panel figure
SAVE_FIGS = False      # write each paper figure to PDF
FIG_DIR = "."

# Colorblind-safe (Okabe-Ito) colors for illumination groups j = 1, 2, 3, ...
GROUP_COLORS = ["#0072B2", "#E69F00", "#8C8C8C", "#009E73", "#CC79A7"]
TURNOFF_COLOR = "#D55E00"


def group_colors(group):
    """Per-point colors for grouping indices (0-based in the linearizer)."""
    return np.asarray(GROUP_COLORS)[np.asarray(group, dtype=int) % len(GROUP_COLORS)]


def group_handles(group):
    """Legend handles labelled with the paper's group index j (1-based)."""
    return [
        Line2D([], [], ls="", marker="o", ms=3, mec="none",
               mfc=GROUP_COLORS[g % len(GROUP_COLORS)], label=rf"$j = {g + 1}$")
        for g in np.unique(np.asarray(group, dtype=int))
    ]


def save_fig(fig, name):
    if SAVE_FIGS:
        fig.savefig(f"{FIG_DIR}/{name}_det{DETECTOR:03d}.pdf")


butler = Butler("main")
camera = butler.get("camera", instrument="LSSTCam", collections="LSSTCam/defaults")

DETECTOR = 94 # 15
AMP = "C03"  # "C16"

In [2]:
linearizer = butler.get("linearizer", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/calib")
ptc = butler.get("ptc", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/calib")


In [ ]:
linTurnoffs = []
types = []

ampName = 'C07'

from matplotlib.colors import ListedColormap

colors3 = ListedColormap(['cornflowerblue', 'darkorange', 'gray'])

lin = butler.get("linearizer", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/calib")
# ptc = butler.get("linearizerPtc", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/calib")

# Stacked so it fits one LaTeX column at 12 pt.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(COLUMN_WIDTH, 1.35 * COLUMN_WIDTH), layout="constrained")

# --- Top panel: linearity turnoff ---

types.append(camera[DETECTOR].getPhysicalType())
linTurnoff = float(lin.linearityTurnoff[ampName])
linTurnoffs.append(linTurnoff)

ax1.scatter(lin.inputAbscissa[ampName], lin.inputOrdinate[ampName], s=3, c=linearizer.inputGroupingIndex[ampName], cmap=colors3)
ax1.axhline(linTurnoff, linestyle="-", color="r", alpha=0.25, linewidth=0.75)
ax1.text(
    5, linTurnoff*.98,
    r"$\mu > 1.05 \times 10^5$ ADU", color="r", alpha=0.25, va="top",
)

ax1.set_xlim(0, 7e1)
ax1.set_ylim(0, 1.2e5)
# ax1.set_yscale('log')
# ax1.set_xscale('log')
ax1.ticklabel_format(style='sci', axis='y', scilimits=(0, 0), useMathText=True)
ax1.set_xlabel(r"Abscissa, $\Delta t$ (s)")
ax1.set_ylabel(r"Ordinate, $\mu_{\mathrm{ref}}$ (ADU)")
ax1.axhspan(linTurnoff, ax1.get_ylim()[1], facecolor='none', edgecolor='red', hatch='\\', linewidth=0, alpha=0.25)


ax1.text(32.5, 0.03e5, r"$k_1$", color="cornflowerblue")
ax1.text(15, 0.175e5, r"$k_2$", color="darkorange")
ax1.text(30, 0.45e5, r"$k_3$", color="gray")


# --- Bottom panel: fit residuals ---
ax2.scatter(lin.inputOrdinate[ampName], 1e4 * lin.fitResiduals[ampName]/lin.inputOrdinate[ampName], s=3, c=linearizer.inputGroupingIndex[ampName], cmap=colors3)
ax2.axhline(0, linestyle="--", color="k", alpha=0.5)
ax2.ticklabel_format(style='sci', axis='both', scilimits=(0, 0), useMathText=True)
ax2.set_ylim(-2.0, 2.0)
ax2.set_xlabel(r"$\mu_{\mathrm{ref}}$ (ADU)")
ax2.set_ylabel(r"$(r_i - 1) \times 10^4$")

fig.align_ylabels((ax1, ax2))
plt.savefig("reference-amp-non-linearity_det94.pdf")

In [ ]:
# --- Paper figure: double-spline non-linearity, S_abs(mu) (top) and S_rel(mu) (bottom) ---
from scipy.interpolate import Akima1DInterpolator


def unpack_double_spline(coeffs):
    """coeffs = [nRel, nAbs, relNodes, relValues, absNodes, absValues, (padding)]"""
    nRel, nAbs = int(coeffs[0]), int(coeffs[1])
    rel = np.split(np.asarray(coeffs[2:2 + 2*nRel]), 2) if nRel > 0 else None
    s = 2 + 2*nRel
    absol = np.split(np.asarray(coeffs[s:s + 2*nAbs]), 2) if nAbs > 0 else None
    return rel, absol


def eval_spline(nodes, values, x):
    # Same evaluation as ip_isr LinearizeDoubleSpline (Akima, clipped to node range)
    return Akima1DInterpolator(nodes, values, method="akima")(np.clip(x, nodes[0], nodes[-1]))


refAmp = linearizer.absoluteReferenceAmplifier

fig, (axAbs, axRel) = plt.subplots(2, 1, figsize=(COLUMN_WIDTH, 1.35 * COLUMN_WIDTH), sharex=True,
                                   layout="constrained")

# --- Top: absolute non-linearity S_abs(mu) on the reference amplifier ---
_, (absNodes, absValues) = unpack_double_spline(linearizer.linearityCoeffs[refAmp])
xs = np.linspace(absNodes[0], absNodes[-1], 2000)

mu = np.asarray(linearizer.inputOrdinate[refAmp])
resid = np.asarray(linearizer.fitResiduals[refAmp])  # NaN where rejected by the fit
group = np.asarray(linearizer.inputGroupingIndex[refAmp])
good = np.isfinite(resid) & (mu > 0)
sMeas = eval_spline(absNodes, absValues, mu[good]) + resid[good]  # per-exposure measurement

axAbs.axhline(0, color="k", lw=0.6, ls="--", alpha=0.6, zorder=1)
# Points: large, semi-transparent, group-colored, drawn underneath the fit.
axAbs.scatter(mu[good], sMeas, s=12, c="gray", lw=0, alpha=0.45,
              rasterized=True, zorder=2)
# Spline: thin black line on top, so the scatter shows as a halo around it.
axAbs.plot(xs, eval_spline(absNodes, absValues, xs), color="k", lw=0.7, zorder=3)
# Nodes: small open markers, drawn last.
axAbs.plot(absNodes, absValues, ls="", marker="o", ms=3, mfc="white", mec="k", mew=0.7, zorder=4)
axAbs.set_ylabel(r"$S_{\mathrm{abs}}(\mu)$ (ADU)")
axAbs.set_xlim(0, )

# --- Bottom: relative non-linearity S_rel(mu) of every other amplifier w.r.t. refAmp ---
# Shade encodes segment number; blue = C1x row, orange = C0x row.
halfCmaps = {"1": plt.cm.Blues, "0": plt.cm.Oranges}
plotted = {"1": [], "0": []}

axRel.axhline(0, color="k", lw=0.6, ls="--", alpha=0.6, zorder=1)
for amp in [a.getName() for a in camera[DETECTOR]]:
    if amp == refAmp or linearizer.linearityType[amp] != "DoubleSpline":
        continue
    rel, _ = unpack_double_spline(linearizer.linearityCoeffs[amp])
    if rel is None:
        continue
    relNodes, relValues = rel
    xs = np.linspace(relNodes[0], relNodes[-1], 2000)
    half, seg = amp[1], int(amp[2])
    color = halfCmaps[half](0.45 + 0.5 * seg / 7)
    axRel.plot(xs, eval_spline(relNodes, relValues, xs), color=color, lw=0.9, zorder=2)
    axRel.plot(relNodes, relValues, ls="", marker="o", ms=2.5, mfc=color, mec=color, mew=0.7, zorder=3)
    plotted[half].append(amp)

axRel.set_xlabel(r"$\mu_i$ (ADU)")
axRel.set_ylabel(r"$S_{\mathrm{rel}}(\mu)$ (ADU)")
handles = [
    Line2D([], [], color=halfCmaps[h](0.7), lw=1.2, label=f"{min(a)}–{max(a)}")
    for h, a in plotted.items() if a
]
axRel.legend(handles=handles, loc="best")

for ax in (axAbs, axRel):
    ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)

fig.align_ylabels((axAbs, axRel))
plt.savefig("abs-rel-non-linearity_det94.pdf")